In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/prathvichauhan/movieds/IMDB Dataset.csv')

In [ ]:
!pip install -q contractions spacy transformers tokenizers gensim
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 62.5 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import pandas as pd
import numpy as np
import re
import contractions

import nltk
import spacy

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

from collections import Counter

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

nlp = spacy.load("en_core_web_sm")

print("Libraries loaded successfully!")

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Libraries loaded successfully!


In [ ]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/prathvichauhan/movieds/IMDB Dataset.csv


In [ ]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nSentiment distribution:")
print(df['sentiment'].value_counts())

Shape: (50000, 2)

Columns:
Index(['review', 'sentiment'], dtype='object')

Missing values:
review       0
sentiment    0
dtype: int64

Duplicate rows:
418

Sentiment distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [ ]:
pd.set_option('display.max_colwidth', 300)

df.head(5)

,review,sentiment
0,"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO....",positive
1,"A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />The actors are extremely well chosen- Michael Sheen not only ""has got all the pola...",positive
2,"I thought this was a wonderful way to spend time on a too hot summer weekend, sitting in the air conditioned theater and watching a light-hearted comedy. The plot is simplistic, but the dialogue is witty and the characters are likable (even the well bread suspected serial killer). While some may...",positive
3,"Basically there's a family where a little boy (Jake) thinks there's a zombie in his closet & his parents are fighting all the time.<br /><br />This movie is slower than a soap opera... and suddenly, Jake decides to become Rambo and kill the zombie.<br /><br />OK, first of all when you're going t...",negative
4,"Petter Mattei's ""Love in the Time of Money"" is a visually stunning film to watch. Mr. Mattei offers us a vivid portrait about human relations. This is a movie that seems to be telling us what money, power and success do to people in the different situations we encounter. <br /><br />This being a...",positive


In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (49582, 2)


In [ ]:
print(df.loc[0, 'review'])

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fac

# Basic Preprocessing with lowercasing the Words, Expanding contractions

In [ ]:
def preprocess_basic(text):

    # Lowercase
    text = text.lower()

    # Remove HTML
    text = re.sub(r'<.*?>', ' ', text)

    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # Expand contractions
    text = contractions.fix(text)

    # Remove email
    text = re.sub(r'\S+@\S+', ' ', text)

    # Remove numbers
    text = re.sub(r'\d+', ' ', text)

    # Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df['cleaned_text'] = df['review'].apply(preprocess_basic)

df[['review', 'cleaned_text']].head()

,review,cleaned_text
0,"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO....",one of the other reviewers has mentioned that after watching just oz episode you will be hooked they are right as this is exactly what happened with me the first thing that struck me about oz was its brutality and unflinching scenes of violence which set in right from the word go trust me this i...
1,"A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />The actors are extremely well chosen- Michael Sheen not only ""has got all the pola...",a wonderful little production the filming technique is very unassuming very old time bbc fashion and gives a comforting and sometimes discomforting sense of realism to the entire piece the actors are extremely well chosen michael sheen not only has got all the polari but he has all the voices do...
2,"I thought this was a wonderful way to spend time on a too hot summer weekend, sitting in the air conditioned theater and watching a light-hearted comedy. The plot is simplistic, but the dialogue is witty and the characters are likable (even the well bread suspected serial killer). While some may...",i thought this was a wonderful way to spend time on a too hot summer weekend sitting in the air conditioned theater and watching a light hearted comedy the plot is simplistic but the dialogue is witty and the characters are likable even the well bread suspected serial killer while some may be di...
3,"Basically there's a family where a little boy (Jake) thinks there's a zombie in his closet & his parents are fighting all the time.<br /><br />This movie is slower than a soap opera... and suddenly, Jake decides to become Rambo and kill the zombie.<br /><br />OK, first of all when you're going t...",basically there is a family where a little boy jake thinks there is a zombie in his closet his parents are fighting all the time this movie is slower than a soap opera and suddenly jake decides to become rambo and kill the zombie ok first of all when you are going to make a film you must decide ...
4,"Petter Mattei's ""Love in the Time of Money"" is a visually stunning film to watch. Mr. Mattei offers us a vivid portrait about human relations. This is a movie that seems to be telling us what money, power and success do to people in the different situations we encounter. <br /><br />This being a...",petter mattei s love in the time of money is a visually stunning film to watch mr mattei offers us a vivid portrait about human relations this is a movie that seems to be telling us what money power and success do to people in the different situations we encounter this being a variation on the a...


## Removing Stop words

In [ ]:
stop_words = set(stopwords.words('english'))

print("Number of stopwords:", len(stop_words))
print(list(stop_words)[:20])

Number of stopwords: 198
['again', 'ourselves', 'on', 'as', 'herself', 'not', 'who', 'yourselves', 'myself', 'was', 'mightn', 'it', 'ain', 'me', "they'll", "they've", 'why', 'll', 'their', 'up']


## Word Tokenization (From NLTK)

In [ ]:
df['nltk_tokens'] = df['cleaned_text'].apply(word_tokenize)

df[['cleaned_text', 'nltk_tokens']].head()

,cleaned_text,nltk_tokens
0,one of the other reviewers has mentioned that after watching just oz episode you will be hooked they are right as this is exactly what happened with me the first thing that struck me about oz was its brutality and unflinching scenes of violence which set in right from the word go trust me this i...,"[one, of, the, other, reviewers, has, mentioned, that, after, watching, just, oz, episode, you, will, be, hooked, they, are, right, as, this, is, exactly, what, happened, with, me, the, first, thing, that, struck, me, about, oz, was, its, brutality, and, unflinching, scenes, of, violence, which,..."
1,a wonderful little production the filming technique is very unassuming very old time bbc fashion and gives a comforting and sometimes discomforting sense of realism to the entire piece the actors are extremely well chosen michael sheen not only has got all the polari but he has all the voices do...,"[a, wonderful, little, production, the, filming, technique, is, very, unassuming, very, old, time, bbc, fashion, and, gives, a, comforting, and, sometimes, discomforting, sense, of, realism, to, the, entire, piece, the, actors, are, extremely, well, chosen, michael, sheen, not, only, has, got, a..."
2,i thought this was a wonderful way to spend time on a too hot summer weekend sitting in the air conditioned theater and watching a light hearted comedy the plot is simplistic but the dialogue is witty and the characters are likable even the well bread suspected serial killer while some may be di...,"[i, thought, this, was, a, wonderful, way, to, spend, time, on, a, too, hot, summer, weekend, sitting, in, the, air, conditioned, theater, and, watching, a, light, hearted, comedy, the, plot, is, simplistic, but, the, dialogue, is, witty, and, the, characters, are, likable, even, the, well, brea..."
3,basically there is a family where a little boy jake thinks there is a zombie in his closet his parents are fighting all the time this movie is slower than a soap opera and suddenly jake decides to become rambo and kill the zombie ok first of all when you are going to make a film you must decide ...,"[basically, there, is, a, family, where, a, little, boy, jake, thinks, there, is, a, zombie, in, his, closet, his, parents, are, fighting, all, the, time, this, movie, is, slower, than, a, soap, opera, and, suddenly, jake, decides, to, become, rambo, and, kill, the, zombie, ok, first, of, all, w..."
4,petter mattei s love in the time of money is a visually stunning film to watch mr mattei offers us a vivid portrait about human relations this is a movie that seems to be telling us what money power and success do to people in the different situations we encounter this being a variation on the a...,"[petter, mattei, s, love, in, the, time, of, money, is, a, visually, stunning, film, to, watch, mr, mattei, offers, us, a, vivid, portrait, about, human, relations, this, is, a, movie, that, seems, to, be, telling, us, what, money, power, and, success, do, to, people, in, the, different, situati..."


In [ ]:
def remove_stopwords(tokens):
    return [
        word for word in tokens
        if word.lower() not in stop_words
    ]

df['tokens_no_stopwords'] = df['nltk_tokens'].apply(remove_stopwords)

df[['nltk_tokens', 'tokens_no_stopwords']].head()

,nltk_tokens,tokens_no_stopwords
0,"[one, of, the, other, reviewers, has, mentioned, that, after, watching, just, oz, episode, you, will, be, hooked, they, are, right, as, this, is, exactly, what, happened, with, me, the, first, thing, that, struck, me, about, oz, was, its, brutality, and, unflinching, scenes, of, violence, which,...","[one, reviewers, mentioned, watching, oz, episode, hooked, right, exactly, happened, first, thing, struck, oz, brutality, unflinching, scenes, violence, set, right, word, go, trust, show, faint, hearted, timid, show, pulls, punches, regards, drugs, sex, violence, hardcore, classic, use, word, ca..."
1,"[a, wonderful, little, production, the, filming, technique, is, very, unassuming, very, old, time, bbc, fashion, and, gives, a, comforting, and, sometimes, discomforting, sense, of, realism, to, the, entire, piece, the, actors, are, extremely, well, chosen, michael, sheen, not, only, has, got, a...","[wonderful, little, production, filming, technique, unassuming, old, time, bbc, fashion, gives, comforting, sometimes, discomforting, sense, realism, entire, piece, actors, extremely, well, chosen, michael, sheen, got, polari, voices, pat, truly, see, seamless, editing, guided, references, willi..."
2,"[i, thought, this, was, a, wonderful, way, to, spend, time, on, a, too, hot, summer, weekend, sitting, in, the, air, conditioned, theater, and, watching, a, light, hearted, comedy, the, plot, is, simplistic, but, the, dialogue, is, witty, and, the, characters, are, likable, even, the, well, brea...","[thought, wonderful, way, spend, time, hot, summer, weekend, sitting, air, conditioned, theater, watching, light, hearted, comedy, plot, simplistic, dialogue, witty, characters, likable, even, well, bread, suspected, serial, killer, may, disappointed, realize, match, point, risk, addiction, thou..."
3,"[basically, there, is, a, family, where, a, little, boy, jake, thinks, there, is, a, zombie, in, his, closet, his, parents, are, fighting, all, the, time, this, movie, is, slower, than, a, soap, opera, and, suddenly, jake, decides, to, become, rambo, and, kill, the, zombie, ok, first, of, all, w...","[basically, family, little, boy, jake, thinks, zombie, closet, parents, fighting, time, movie, slower, soap, opera, suddenly, jake, decides, become, rambo, kill, zombie, ok, first, going, make, film, must, decide, thriller, drama, drama, movie, watchable, parents, divorcing, arguing, like, real,..."
4,"[petter, mattei, s, love, in, the, time, of, money, is, a, visually, stunning, film, to, watch, mr, mattei, offers, us, a, vivid, portrait, about, human, relations, this, is, a, movie, that, seems, to, be, telling, us, what, money, power, and, success, do, to, people, in, the, different, situati...","[petter, mattei, love, time, money, visually, stunning, film, watch, mr, mattei, offers, us, vivid, portrait, human, relations, movie, seems, telling, us, money, power, success, people, different, situations, encounter, variation, arthur, schnitzler, play, theme, director, transfers, action, pre..."


## STEMMING

In [ ]:
stemmer = PorterStemmer()

def stem_tokens(tokens):
    return [
        stemmer.stem(word)
        for word in tokens
    ]

df['stemmed_tokens'] = df['tokens_no_stopwords'].apply(
    stem_tokens
)

df[['tokens_no_stopwords', 'stemmed_tokens']].head()

,tokens_no_stopwords,stemmed_tokens
0,"[one, reviewers, mentioned, watching, oz, episode, hooked, right, exactly, happened, first, thing, struck, oz, brutality, unflinching, scenes, violence, set, right, word, go, trust, show, faint, hearted, timid, show, pulls, punches, regards, drugs, sex, violence, hardcore, classic, use, word, ca...","[one, review, mention, watch, oz, episod, hook, right, exactli, happen, first, thing, struck, oz, brutal, unflinch, scene, violenc, set, right, word, go, trust, show, faint, heart, timid, show, pull, punch, regard, drug, sex, violenc, hardcor, classic, use, word, call, oz, nicknam, given, oswald..."
1,"[wonderful, little, production, filming, technique, unassuming, old, time, bbc, fashion, gives, comforting, sometimes, discomforting, sense, realism, entire, piece, actors, extremely, well, chosen, michael, sheen, got, polari, voices, pat, truly, see, seamless, editing, guided, references, willi...","[wonder, littl, product, film, techniqu, unassum, old, time, bbc, fashion, give, comfort, sometim, discomfort, sens, realism, entir, piec, actor, extrem, well, chosen, michael, sheen, got, polari, voic, pat, truli, see, seamless, edit, guid, refer, william, diari, entri, well, worth, watch, terr..."
2,"[thought, wonderful, way, spend, time, hot, summer, weekend, sitting, air, conditioned, theater, watching, light, hearted, comedy, plot, simplistic, dialogue, witty, characters, likable, even, well, bread, suspected, serial, killer, may, disappointed, realize, match, point, risk, addiction, thou...","[thought, wonder, way, spend, time, hot, summer, weekend, sit, air, condit, theater, watch, light, heart, comedi, plot, simplist, dialogu, witti, charact, likabl, even, well, bread, suspect, serial, killer, may, disappoint, realiz, match, point, risk, addict, thought, proof, woodi, allen, still,..."
3,"[basically, family, little, boy, jake, thinks, zombie, closet, parents, fighting, time, movie, slower, soap, opera, suddenly, jake, decides, become, rambo, kill, zombie, ok, first, going, make, film, must, decide, thriller, drama, drama, movie, watchable, parents, divorcing, arguing, like, real,...","[basic, famili, littl, boy, jake, think, zombi, closet, parent, fight, time, movi, slower, soap, opera, suddenli, jake, decid, becom, rambo, kill, zombi, ok, first, go, make, film, must, decid, thriller, drama, drama, movi, watchabl, parent, divorc, argu, like, real, life, jake, closet, total, r..."
4,"[petter, mattei, love, time, money, visually, stunning, film, watch, mr, mattei, offers, us, vivid, portrait, human, relations, movie, seems, telling, us, money, power, success, people, different, situations, encounter, variation, arthur, schnitzler, play, theme, director, transfers, action, pre...","[petter, mattei, love, time, money, visual, stun, film, watch, mr, mattei, offer, us, vivid, portrait, human, relat, movi, seem, tell, us, money, power, success, peopl, differ, situat, encount, variat, arthur, schnitzler, play, theme, director, transfer, action, present, time, new, york, differ,..."


In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer

In [ ]:
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=5000,
    special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]"]
)

tokenizer.train_from_iterator(
    df['cleaned_text'].head(10000),
    trainer=trainer
)

## BPE

In [ ]:
sample_text = df.loc[0, 'cleaned_text']

encoding = tokenizer.encode(sample_text)

print("Original:")
print(sample_text[:500])

print("\nBPE Tokens:")
print(encoding.tokens[:50])

Original:
one of the other reviewers has mentioned that after watching just oz episode you will be hooked they are right as this is exactly what happened with me the first thing that struck me about oz was its brutality and unflinching scenes of violence which set in right from the word go trust me this is not a show for the faint hearted or timid this show pulls no punches with regards to drugs sex or violence its is hardcore in the classic use of the word it is called oz as that is the nickname given to

BPE Tokens:
['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', 'oz', 'episode', 'you', 'will', 'be', 'hook', 'ed', 'they', 'are', 'right', 'as', 'this', 'is', 'exactly', 'what', 'happened', 'with', 'me', 'the', 'first', 'thing', 'that', 'stru', 'ck', 'me', 'about', 'oz', 'was', 'its', 'brut', 'ality', 'and', 'un', 'fl', 'in', 'ching', 'scenes', 'of', 'violence']


In [ ]:
from transformers import AutoTokenizer

bert_tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Word Piece

In [ ]:
encoding = bert_tokenizer(
    sample_text,
    add_special_tokens=True
)

tokens = bert_tokenizer.convert_ids_to_tokens(
    encoding['input_ids']
)

print(tokens[:50])

['[CLS]', 'one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', 'oz', 'episode', 'you', 'will', 'be', 'hooked', 'they', 'are', 'right', 'as', 'this', 'is', 'exactly', 'what', 'happened', 'with', 'me', 'the', 'first', 'thing', 'that', 'struck', 'me', 'about', 'oz', 'was', 'its', 'brutality', 'and', 'un', '##fl', '##in', '##ching', 'scenes', 'of', 'violence', 'which', 'set']


In [ ]:
from gensim.models import Word2Vec

## Word2Vec

In [ ]:
sentences = df['tokens_no_stopwords'].tolist()

word2vec_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4
)

print("Vocabulary size:",
      len(word2vec_model.wv))

word = "movie"

vector = word2vec_model.wv[word]

print("Vector size:", vector.shape)
print(vector[:10])

word2vec_model.wv.most_similar("movie", topn=10)

Vocabulary size: 38927
Vector size: (100,)
[-0.70673394 -1.3920724  -0.05127436 -0.32496473 -0.16126326 -0.35052368
  1.0732639   1.2628319   0.45765948 -1.0793014 ]


[('film', 0.8290221691131592),
 ('flick', 0.6595902442932129),
 ('movies', 0.6488916873931885),
 ('sure', 0.5672906637191772),
 ('think', 0.5633909702301025),
 ('guess', 0.5434600710868835),
 ('anyway', 0.5351114869117737),
 ('really', 0.5283700227737427),
 ('sequel', 0.5244483947753906),
 ('anyways', 0.519897997379303)]